In [2]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - old folder structure
# Gabrielle Baxter 08/07/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np

# CHANGED FOR WRAPPER: run_heal_pipeline.sh sets the HEAL_PARENT environment
# variable, otherwise the hard-coded default below is used (no trailing /)
parent_folder = os.environ.get('HEAL_PARENT', '/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075').rstrip('/')
print('Parent folder:', parent_folder)
parent_nifti_folder = os.path.join(parent_folder,'derivatives/all')
maps_folder = os.path.join(parent_folder,'maps')
print(maps_folder)

diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
print(diffusion_times)

fixed = ants.image_read(os.path.join(parent_nifti_folder,diffusion_times[0] + '/dwiec.nii'))
print('Fixed',os.path.join(parent_nifti_folder,diffusion_times[0] + '/dwiec.nii'))
fixed_array = fixed.numpy()
fixed_b0= fixed_array[:,:,:,0]
fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

for dt in diffusion_times[1:]: #iterate through the other diffusion times registering to lowest
    print('Moving',os.path.join(parent_nifti_folder, dt + '/dwiec.nii'))
    moving = ants.image_read(os.path.join(parent_nifti_folder, dt + '/dwiec.nii'))
    moving_array = moving.numpy()
    nx,ny,nz,nvols = np.shape(moving_array)

    # Register dwi - not really necessary but worth doing
    registered_dwi = np.zeros([nx,ny,nz,nvols])
    save_filename = os.path.join(parent_nifti_folder, dt + '/dwiec_reg.nii')
    if not os.path.exists(save_filename):
        # register dwi
        for vol in range(0,nvols): # register all the volumes of the dwi for each diffusion time to the b0 of the lowest diffusion time image
            moving_temp = moving_array[:,:,:,vol]
            moving_ants = ants.from_numpy(moving_temp, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='SyN')  
            warped_image = ants.apply_transforms(
                fixed=fixed_ants,
                moving=moving_ants,
                transformlist=reg['fwdtransforms'],
                interpolator='bSpline'  
            )
            warped_image_array = warped_image.numpy()
            registered_dwi[:,:,:,vol] = warped_image_array

        registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
        registered_dwi_ants.to_filename(save_filename)

    # Register maps
    moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
    moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
    reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='SyN')    
    
    maps = ['AD','RD','FA','L2','L3']
    for map in maps:
        current_map = ants.image_read(os.path.join(maps_folder,dt + '_' + map + '.nii.gz'))
        map_array = current_map.numpy()
        print(os.path.join(maps_folder,dt + '_' + map + '.nii.gz'))

        warped_map = ants.apply_transforms(
            fixed=fixed_ants,
            moving=current_map,
            transformlist=reg['fwdtransforms'],
            interpolator='bSpline'  
        )
        save_filename = os.path.join(maps_folder,dt + '_' + map + '_withinscanreg.nii.gz')
        warped_map.to_filename(save_filename)


/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps
['0022ms', '0042ms', '0081ms', '0156ms', '0300ms']
Fixed /Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/derivatives/all/0022ms/dwiec.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/derivatives/all/0042ms/dwiec.nii
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0042ms_AD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0042ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0042ms_FA.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0042ms_L2.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0042ms_L3.nii.gz
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/derivatives/all/0081ms/dwiec.nii
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0081ms_AD.nii.gz
/Users/gabriellebaxter/Documents/